<a href="https://colab.research.google.com/github/souro26/bayesian-a-b-testing/blob/main/examples/03_drug_dosage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
try:
    import argonx  # noqa: F401
except ImportError:
    !pip install -q git+https://github.com/souro26/bayesian-a-b-testing.git

# Example 03 — Drug Dosage & Outlier-Robust Inference

Clinical trials often deal with 'messy' patient outcomes. While most patients show a standard response to a drug, a few 'super-responders' or patients with extreme reactions can create massive outliers.

### The Scientific Problem
Standard statistical tests (like t-tests or Gaussian Bayesian models) assume that noise is normally distributed. This means they are highly sensitive to outliers. A single extreme value can shift the estimated mean significantly, potentially leading a researcher to believe a drug is effective when it isn't—or vice versa.

### The Solution: Student-T Likelihood
The **Student-T** distribution has 'fat tails'. It treats extreme values as expected (though rare) events, preventing them from 'pulling' the mean away from the bulk of the data. This provides a much more robust estimate of the true treatment effect.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from argonx import Experiment

plt.style.use("seaborn-v0_8-muted")
sns.set_theme(style="whitegrid")

## 1. Creating the 'Tainted' Data

We'll simulate two groups with the exact same true mean (20 units of pain reduction). However, we'll add 5 'super-responders' to the Control group who show a massive 80-point reduction.

In [ ]:
np.random.seed(1337)
N = 250

# True effect: 20 for both
control = np.random.normal(20, 5, N)
variant = np.random.normal(20, 5, N)

# Add massive outliers to control only
control[0:5] = [85, 90, 78, 82, 88]

df = pd.DataFrame(
    {
        "group": ["control"] * N + ["new_dose"] * N,
        "pain_reduction": np.concatenate([control, variant]),
    }
)

plt.figure(figsize=(10, 5))
sns.boxplot(x="group", y="pain_reduction", data=df, palette=["#94A3B8", "#2563EB"])
plt.title("Outliers in Clinical Data", fontsize=14, fontweight="bold")
plt.ylabel("Pain Reduction Score", fontsize=12)
plt.show()

Notice the cluster of points at the top of the 'control' column. These outliers are significantly higher than the rest of the data.

## 2. The Gaussian Trap

First, we run a standard Gaussian model. Watch how it gets 'fooled' by the outliers into thinking the Control is superior.

In [ ]:
exp_gauss = Experiment(
    data=df,
    variant_col="group",
    primary_metric="pain_reduction",
    model="gaussian",
    control="control",
)
res_gauss = exp_gauss.run()

print(f"Gaussian P(control is best): {res_gauss.prob_best['control']:.1%}")
print(f"Gaussian Mean Lift:          {res_gauss.lift['mean']:.2%}")

## 3. The Robust Student-T Alternative

Now we switch to the `studentt` model. This model effectively 'down-weights' the influence of the outliers.

In [ ]:
exp_robust = Experiment(
    data=df,
    variant_col="group",
    primary_metric="pain_reduction",
    model="studentt",
    control="control",
)
res_robust = exp_robust.run()

print(f"Robust P(control is best): {res_robust.prob_best['control']:.1%}")
print(f"Robust Mean Lift:          {res_robust.lift['mean']:.2%}")

## 4. Visual Comparison

In [ ]:
res_robust.plot(
    metric_name="Pain Reduction Score", suptitle="Outlier-Robust Decision Dashboard"
)

### Conclusion

The difference is stark:
*   The **Gaussian** model is biased by just 5 outliers (2% of data), incorrectly favoring the control group.
*   The **Student-T** model sees through the noise, correctly identifying that the two groups are functionally equivalent.

In high-stakes environments like healthcare or clinical trials, using a robust model like Student-T can be the difference between a failed research project and a successful, accurate discovery.